# BrainGFM Demo: Mask → T1 Image

Minimal test: brain mask (no lesion) → one **T1** slice.

Needs a local / server **CUDA GPU**. No Google account required.

Mask: `samples/mask_testing/input/brain.nii.gz` (`0` background, `1` brain).

In [ ]:
import sys
from pathlib import Path

IN_COLAB = Path("/content").exists()
REPO_URL = "https://github.com/YutingHe-list/BrainGFM.git"
REPO_DIR = Path("/content/BrainGFM") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not (REPO_DIR / "lora" / "infer_online.py").is_file():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    if (Path.cwd().parent / "lora" / "infer_online.py").is_file():
        %cd ..
    REPO_DIR = Path.cwd()

assert (REPO_DIR / "lora" / "infer_online.py").is_file(), Path.cwd()
print("repo:", REPO_DIR.resolve())

In [ ]:
import subprocess

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import torch
except ImportError:
    pip_install("torch", "torchvision") if IN_COLAB else pip_install("-r", "requirements.txt")

pip_install("nibabel", "huggingface_hub", "matplotlib", "pillow", "ruamel.yaml", "safetensors")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Need a CUDA GPU."

In [ ]:
MODALITY = "t1"
MASK_PATH = "samples/mask_testing/input/brain.nii.gz"
OUTPUT_PATH = "samples/mask_testing/output/brain_t1.nii.gz"

!python lora/infer_online.py --input_path {MASK_PATH} --output_path {OUTPUT_PATH} --modality {MODALITY}

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

stem = OUTPUT_PATH[:-7]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(Image.open(stem + "_mask.png"))
axes[0].set_title("Mask (no lesion)")
axes[1].imshow(Image.open(stem + "_image.png"), cmap="gray")
axes[1].set_title("Generated T1")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print("saved:", Path(OUTPUT_PATH).resolve())